## Notebook 01_ingestao_bronze — Ingestão de dados do Datajud



#### Objetivo
Este notebook realiza a coleta dos processos da Recuperação Judicial,
Recuperação Extrajudicial e Falência diretamente da API pública do Datajud (CNJ) e os
persiste na camada **Bronze** do lakehouse (`default.bronze_datajud_raw`), sem qualquer
transformação — o dado bruto é o "cofre de evidências" da arquitetura medalhão.

#### Entradas e saídas
- **Entrada**: endpoints públicos `https://api-publica.datajud.cnj.jus.br/api_publica_{tribunal}/_search` (27 tribunais estaduais + TJDFT)
- **Saída**: tabela Delta `default.bronze_datajud_raw` com 3 colunas:
  - `raw_payload` (string) — o JSON do processo exatamente como recebido da API
  - `tribunal_origem` (string) — metadado de fonte
  - `ingestion_timestamp` (timestamp) — metadado de controle da carga

#### Decisões de desenho
1. **Recorte definido pelas perguntas de negócio**: filtro por assunto pai 4993 - recuperação judicial, falência. Esse assunto pai engloba os assuntos filhos: 4992 -faLência (código antigo); 4994 recuperação extrajudicial; 4995 - concordata preventiva (código antigo); 4996 concordata suspensiva (código antigo),;  4997 falência fraudulenta (código antigo);

2. **Recorte definido pelo tempo para o download**: `dataAjuizamento >= 01/01/1995`. Mais tarde decidiu-se analisar processos a partir de 9/06/2005 (vigência da atual Lei 11.101/2005 que regula a recuperação judicial, extrajudicial e a falência de empresários e sociedades empresárias). 

3. **Dado bruto imutável dentro do registro**: o JSON é salvo como string, sem tipagem nem
   flattening — toda estruturação acontece na Silver.

4. **Persistência incremental por tribunal**: cada tribunal é gravado em lote ao término da
   sua paginação, liberando a memória do driver antes do próximo — evita OOM em coletas longas.

5. **Full-refresh da rotina**: o `DROP TABLE IF EXISTS` no início do código significa que a execução
   completa reconstrói a Bronze inteira. Esse procedimento foi utilizado para viabilizar as múltiplas tentativas de captura dos dados junto ao CNJ; o `append` dentro do loop é apenas o mecanismo de gravação por lote de vários tribunais. 

In [0]:
# Databricks Notebook: 01_ingestao_bronze
import requests
import json
import time
from pyspark.sql.functions import current_timestamp, lit

API_KEY = "cDZHYzlZa0JadVREZDJCendQbXY6SkJlTzNjLV9TRENyQk1RdnFKZGRQdw=="

# Substituição da sigla do DF (tjdft em vez de tjdf)
TRIBUNAIS = [
    "tjsp", "tjrj", "tjmg", "tjrs", "tjpr", "tjsc", "tjba", "tjpe", "tjce", 
    "tjdft", "tjes", "tjgo", "tjmt", "tjms", "tjpa", "tjpb", "tjrn", "tjro", 
    "tjse", "tjto", "tjal", "tjam", "tjap", "tjac", "tjrr", "tjpi", "tjma"
]

headers = {
    "Authorization": f"APIKey {API_KEY}",
    "Content-Type": "application/json"
}

payload_base = {
    "size": 10000,
    "query": {
        "bool": {
            "must": [
                {"range": {"dataAjuizamento": {"gte": "1995-01-01T00:00:00.000Z"}}},
                {"terms": {"assuntos.codigo": [4992, 4993, 4994, 4995, 4996, 4997]}}
            ]
        }
    },
    "sort": [{"@timestamp": {"order": "asc"}}, {"id.keyword": {"order": "asc"}}]
}

MAX_RETRIES = 5
INITIAL_BACKOFF = 3

print("=== INICIANDO INGESTÃO MULTI-TRIBUNAL COM PERSISTÊNCIA EM LOTE ===", flush=True)

# 1. Limpa ou recria a tabela Bronze no início da rotina
spark.sql("DROP TABLE IF EXISTS default.bronze_datajud_raw")

total_geral_processos = 0

for tribunal in TRIBUNAIS:
    url = f"https://api-publica.datajud.cnj.jus.br/api_publica_{tribunal}/_search"
    search_after = None
    page = 1
    total_tribunal = 0
    hits_tribunal_batch = []
    
    print(f"\n--- Processando Tribunal: {tribunal.upper()} ---", flush=True)

    while True:
        payload = payload_base.copy()
        if search_after:
            payload["search_after"] = search_after

        success = False
        print(f"-> [{tribunal.upper()}] Pág {page}... ", end="", flush=True)

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = requests.post(url, json=payload, headers=headers, timeout=(10, 120))

                if response.status_code == 200:
                    success = True
                    print("[OK]", flush=True)
                    break

                elif response.status_code in [429, 500, 502, 503, 504]:
                    wait_time = INITIAL_BACKOFF * (2 ** (attempt - 1))
                    print(f"\n⚠️ HTTP {response.status_code}. Aguardando {wait_time}s... (Tentativa {attempt}/{MAX_RETRIES})", flush=True)
                    time.sleep(wait_time)
                else:
                    print(f"\n❌ Erro HTTP {response.status_code}: {response.text}", flush=True)
                    break

            except (requests.exceptions.RequestException, requests.exceptions.Timeout) as e:
                wait_time = INITIAL_BACKOFF * (2 ** (attempt - 1))
                print(f"\n⚠️ Timeout/Falha Conexão. Aguardando {wait_time}s... (Tentativa {attempt}/{MAX_RETRIES})", flush=True)
                time.sleep(wait_time)

        if not success:
            print(f"\n❌ Falha ao buscar dados do {tribunal.upper()} na página {page}. Avançando para próximo tribunal.", flush=True)
            break

        data = response.json()
        hits = data.get("hits", {}).get("hits", [])

        if not hits:
            print(f"✅ Concluído {tribunal.upper()}! Total do tribunal: {total_tribunal:,} processos.", flush=True)
            break

        for hit in hits:
            hits_tribunal_batch.append((json.dumps(hit["_source"]), tribunal.upper()))

        total_tribunal += len(hits)
        print(f"   L Registros obtidos: +{len(hits)} (Acumulado no tribunal: {total_tribunal:,})", flush=True)

        search_after = hits[-1].get("sort")
        page += 1

        time.sleep(1)

    # 2. Persiste os dados do tribunal corrente no Delta Lake e libera a memória local
    if len(hits_tribunal_batch) > 0:
        df_batch = spark.createDataFrame(hits_tribunal_batch, ["raw_payload", "tribunal_origem"]) \
                        .withColumn("ingestion_timestamp", current_timestamp())
        
        df_batch.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable("default.bronze_datajud_raw")
            
        total_geral_processos += len(hits_tribunal_batch)
        print(f"💾 {len(hits_tribunal_batch):,} registros do {tribunal.upper()} gravados na Bronze.", flush=True)
        
        # Limpa o lote da memória do driver
        del hits_tribunal_batch

print(f"\n=== INGESTÃO FINALIZADA. TOTAL GERAL SALVO NA BRONZE: {total_geral_processos:,} ===", flush=True)

In [0]:
# Databricks Notebook: Ingestao_Complementar_TJMG
import requests
import json
import time
from pyspark.sql.functions import current_timestamp

API_KEY = "cDZHYzlZa0JadVREZDJCendQbXY6SkJlTzNjLV9TRENyQk1RdnFKZGRQdw=="
TRIBUNAL = "tjmg"
URL = f"https://api-publica.datajud.cnj.jus.br/api_publica_{TRIBUNAL}/_search"

headers = {
    "Authorization": f"APIKey {API_KEY}",
    "Content-Type": "application/json"
}

# Redução do tamanho da página para evitar erro 504 no TJMG
PAGE_SIZE = 2500

payload_base = {
    "size": PAGE_SIZE,
    "query": {
        "bool": {
            "must": [
                {"range": {"dataAjuizamento": {"gte": "1995-01-01T00:00:00.000Z"}}},
                {"terms": {"assuntos.codigo": [4992, 4993, 4994, 4995, 4996, 4997]}}
            ]
        }
    },
    "sort": [{"@timestamp": {"order": "asc"}}, {"id.keyword": {"order": "asc"}}]
}

MAX_RETRIES = 5
INITIAL_BACKOFF = 4

print(f"=== INICIANDO INGESTÃO COMPLEMENTAR: {TRIBUNAL.upper()} ===", flush=True)

search_after = None
page = 1
total_tjmg = 0

while True:
    payload = payload_base.copy()
    if search_after:
        payload["search_after"] = search_after

    success = False
    print(f"-> [{TRIBUNAL.upper()}] Pág {page}... ", end="", flush=True)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(URL, json=payload, headers=headers, timeout=(15, 180))

            if response.status_code == 200:
                success = True
                print("[OK]", flush=True)
                break

            elif response.status_code in [429, 500, 502, 503, 504]:
                wait_time = INITIAL_BACKOFF * (2 ** (attempt - 1))
                print(f"\n⚠️ HTTP {response.status_code}. Aguardando {wait_time}s... (Tentativa {attempt}/{MAX_RETRIES})", flush=True)
                time.sleep(wait_time)
            else:
                print(f"\n❌ Erro HTTP {response.status_code}: {response.text}", flush=True)
                break

        except (requests.exceptions.RequestException, requests.exceptions.Timeout) as e:
            wait_time = INITIAL_BACKOFF * (2 ** (attempt - 1))
            print(f"\n⚠️ Timeout/Falha Conexão. Aguardando {wait_time}s... (Tentativa {attempt}/{MAX_RETRIES})", flush=True)
            time.sleep(wait_time)

    if not success:
        print(f"\n❌ Falha persistente ao buscar dados do {TRIBUNAL.upper()} na página {page}.", flush=True)
        break

    data = response.json()
    hits = data.get("hits", {}).get("hits", [])

    if not hits:
        print(f"✅ Concluído {TRIBUNAL.upper()}! Total de processos coletados: {total_tjmg:,}.", flush=True)
        break

    # Converte e grava cada página diretamente no Delta Lake existente em modo APPEND
    hits_batch = [(json.dumps(hit["_source"]), TRIBUNAL.upper()) for hit in hits]
    
    df_page = spark.createDataFrame(hits_batch, ["raw_payload", "tribunal_origem"]) \
                   .withColumn("ingestion_timestamp", current_timestamp())
    
    df_page.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("default.bronze_datajud_raw")

    total_tjmg += len(hits)
    print(f"   L Gravados na Bronze: +{len(hits):,} (Acumulado TJMG: {total_tjmg:,})", flush=True)

    search_after = hits[-1].get("sort")
    page += 1

    del hits_batch
    time.sleep(1)

print(f"\n=== PROCESSAMENTO DO TJMG CONCLUÍDO. REGISTROS ADICIONADOS: {total_tjmg:,} ===", flush=True)

In [0]:
# Validação final da camada Bronze (default.bronze_datajud_raw)
df_resumo = spark.sql("""
    SELECT 
        tribunal_origem, 
        COUNT(*) AS total_processos,
        MIN(ingestion_timestamp) AS primeira_ingestao,
        MAX(ingestion_timestamp) AS ultima_ingestao
    FROM default.bronze_datajud_raw
    GROUP BY tribunal_origem
    ORDER BY total_processos DESC
""")

display(df_resumo)

# Total geral
total_bronze = spark.table("default.bronze_datajud_raw").count()
print(f"Total absoluto de registros na Bronze: {total_bronze:,}")

#### Comentários sobre o código do notebook 01_ingestao_bronze
##### Configuração da coleta

- **Autenticação**: a API do Datajud usa o header `Authorization: APIKey {chave}`.
- **Filtro Elasticsearch**: o corpo da requisição é uma query `bool` com duas cláusulas
  `must` — intervalo de `dataAjuizamento` e lista de códigos de assunto (`terms`).
- **Paginação com `search_after`**: em vez de páginas numeradas (que degradam em bases
  grandes), a API do Datajud usa o cursor `search_after` do Elasticsearch: cada resposta
  retorna um array `sort`, que é repassado na requisição seguinte. O loop `while True`
  encerra quando a página vem vazia (`hits == []`).
- **Ordenação determinística** por `@timestamp` e `id.keyword` — garante que a paginação
  não pule nem repita registros.

##### Resiliência: retry com backoff exponencial

Cada página passa por até `MAX_RETRIES = 5` tentativas. Em erros transitórios (HTTP 429
limite de taxa, 5xx do servidor ou timeout de rede), a rotina aguarda
`INITIAL_BACKOFF * 2^(tentativa-1)` segundos — 3s, 6s, 12s, 24s, 48s — antes de repetir.
Erros definitivos (outros códigos HTTP) interrompem o tribunal corrente e avançam para o
próximo, sem abortar a coleta inteira — o log registra a falha para auditoria.

**Comando-chave**: `requests.post(url, json=payload, headers=headers, timeout=(10, 120))` —
o timeout em tupla separa o tempo de conexão (10s) do tempo de leitura (120s), necessário
páginas de 10.000 registros.

##### Persistência na Bronze (Load)

Cada tribunal finaliza com a gravação do seu lote no Delta Lake:

- `spark.createDataFrame(hits_tribunal_batch, [...])` — materializa as tuplas
  `(payload_json, tribunal)` coletadas no driver;
- `.withColumn("ingestion_timestamp", current_timestamp())` — **metadado de controle**:
  carimbo de data/hora da ingestão em cada registro;
- `.write.format("delta").mode("append").saveAsTable("default.bronze_datajud_raw")` —
  gravação no formato Delta com append; a tabela é criada no primeiro tribunal e
  estendida nos demais;
- `del hits_tribunal_batch` — libera a memória do driver entre lotes, permitindo coletar
  milhões de registros sem estourar o nó.

A camada Bronze está fechada: nenhum notebook das camadas seguintes escreve
nesta tabela — apenas leem.


In [0]:
# ==============================================================================
# QA ADITIVO — valida a Bronze ANTES de seguir para o notebook 002
# (apenas leitura e print — não grava tabelas, não altera o código acima)
# ==============================================================================
bronze_qa = spark.table("default.bronze_datajud_raw")

# 1. Schema esperado (3 colunas, tipos corretos)
esperado = [("raw_payload", "string"), ("tribunal_origem", "string"), ("ingestion_timestamp", "timestamp")]
schema_ok = [(f.name, f.dataType.simpleString()) for f in bronze_qa.schema.fields] == esperado
print(f"[QA 1/5] Schema da Bronze: {'OK' if schema_ok else 'DIVERGENTE -> ' + str([(f.name, f.dataType.simpleString()) for f in bronze_qa.schema.fields])}")

# 2. Nulos nos metadados de controle (esperado: 0)
nulos = bronze_qa.filter(
    bronze_qa.raw_payload.isNull() | bronze_qa.tribunal_origem.isNull() | bronze_qa.ingestion_timestamp.isNull()
).count()
print(f"[QA 2/5] Registros com payload/tribunal/timestamp nulos: {nulos} (esperado: 0)")
assert nulos == 0, "QA falhou: existem nulos nos metadados da Bronze"

# 3. Payload JSON válido em todos os registros (esperado: 0 inválidos)
from pyspark.sql.functions import from_json, col
invalidos = bronze_qa.filter(from_json(col("raw_payload"), "map<string,string>").isNull()).count()
print(f"[QA 3/5] Payloads JSON inválidos: {invalidos} (esperado: 0)")
assert invalidos == 0, "QA falhou: payload bruto não-parseável detectado"

# 4. Contagem por tribunal — cruza com o log da rotina
print("[QA 4/5] Contagem por tribunal na Bronze:")
bronze_qa.groupBy("tribunal_origem").count().orderBy(col("count").desc()).show(30)

# 5. Contagem total de processos
total_bronze = spark.table("default.bronze_datajud_raw").count()
print(f"[QA 5/5] Contagem de processos na Bronze: {total_bronze:,}")

print("\n=== QA DA BRONZE CONCLUÍDO: pronta para leitura pelo 02_silver_transformacao ===")